In [1]:
import pandas as pd
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [2]:
DATA_PATH = "data/imdb_train_with_minor_disease.csv"
df = pd.read_csv(DATA_PATH)

# Clean text (same as before)
df["symptoms"] = df["symptoms"].astype(str).str.lower()

X_text = df["symptoms"]
y = df["disease"]


In [3]:
print(df.columns)
print(df.iloc[0])

Index(['disease', 'symptoms', 'precautions', 'medicine', 'minor_disease'], dtype='object')
disease                                           Fungal infection
symptoms         itching,skin_rash,nodal_skin_eruptions,dischro...
precautions      bath twice, use detol or neem in bathing water...
medicine         Antifungal creams or tablets like Clotrimazole...
minor_disease                                         Skin Allergy
Name: 0, dtype: object


In [4]:
print(df['disease'].value_counts())

disease
Unknown                                    305
Hyperthyroidism                            117
Fungal infection                           116
Pneumonia                                  116
Allergy                                    115
Hepatitis C                                115
Acne                                       114
Hypothyroidism                             114
Hepatitis D                                114
Hepatitis E                                113
Hepatitis B                                113
Arthritis                                  113
Tuberculosis                               113
Chronic cholestasis                        113
Drug Reaction                              113
Gastroenteritis                            112
Dengue                                     112
Dimorphic hemmorhoids(piles)               112
Psoriasis                                  112
AIDS                                       112
GERD                                       112
Chick

In [5]:
print(df.sample(10))

                      disease  \
4628  Urinary tract infection   
277              Heart attack   
4146                Diabetes    
749                      Acne   
4725     Cervical spondylosis   
4493              Common Cold   
4009                  Unknown   
2502      Chronic cholestasis   
2819             Hypoglycemia   
750                      Acne   

                                               symptoms  \
4628  burning_micturition,bladder_discomfort,foul_sm...   
277         vomiting,breathlessness,sweating,chest_pain   
4146  fatigue,weight_loss,restlessness,lethargy,irre...   
749               skin_rash,pus_filled_pimples,scurring   
4725  back_pain,weakness_in_limbs,neck_pain,dizzines...   
4493  continuous_sneezing,chills,fatigue,cough,high_...   
4009  joint_pain,neck_pain,knee_pain,hip_joint_pain,...   
2502  itching,vomiting,yellowish_skin,nausea,loss_of...   
2819  vomiting,fatigue,anxiety,sweating,headache,nau...   
750             skin_rash,pus_filled_pimples,b

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42
)


In [13]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=6000
    )),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])


In [14]:
pipeline.fit(X_train, y_train)
print("✅ Pipeline trained successfully")

✅ Pipeline trained successfully


In [15]:
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"🎯 Model Accuracy: {accuracy * 100:.2f}%")


🎯 Model Accuracy: 93.95%


In [18]:
OUTPUT_DIR = "ml_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

joblib.dump(pipeline, os.path.join(OUTPUT_DIR, "model.pkl"))
print("✅ model.pkl saved successfully (pipeline)")

✅ model.pkl saved successfully (pipeline)


In [19]:
vocab_size = len(pipeline.named_steps["tfidf"].vocabulary_)
print("LOCAL vocab size:", vocab_size)

LOCAL vocab size: 487


In [21]:
model = joblib.load("ml_models/model.pkl")
print(model.predict(["fever headache cough"]))

['Paralysis (brain hemorrhage)']
